In [1]:
# from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
# from rouge_score import rouge_scorer
from pathlib import Path
import sys
import json
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
from peft import PeftModel, PeftConfig
import torch
import gc

RuntimeError: Failed to import transformers.models.bloom.modeling_bloom because of the following error (look up to see its traceback):
This version of jaxlib was built using AVX instructions, which your CPU and/or operating system do not support. You may be able work around this issue by building jaxlib from source.

In [ ]:
gc.collect()
model_name = "meta-llama/Llama-3.2-3B-Instruct"  # or the base model you trained on
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
config = AutoConfig.from_pretrained(model_name)
# manually set rope_scaling to supported structure:
config.rope_scaling = {"type": "dynamic", "factor": 2.0}
config.use_cache = True
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    config=config,
    torch_dtype=torch.float16
)


adapter_path = "./../Training/final_adapter_with_eval_1"  # or wherever your adapter_model.safetensors is
#adapted_model= PeftModel.from_pretrained(base_model,adapter_path)
lora_config = PeftConfig.from_pretrained(adapter_path) 
base_model.add_adapter(lora_config, adapter_name="summarizer")
base_model.disable_adapters()

